
Fraude en Tarjetas de Crédito
El fraude financiero, específicamente el uso no autorizado de tarjetas de crédito o débito, representa pérdidas de miles de millones de dólares anualmente para bancos, comercios y consumidores.

El reto principal para las instituciones financieras es identificar las transacciones fraudulentas en tiempo real o casi real para bloquearlas antes de que se complete el pago, pero sin afectar la experiencia de los clientes legítimos
---



In [3]:
# ==========================================
# STACKING ENSEMBLE PARA DETECCIÓN DE FRAUDE
# ==========================================

# Librerías
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# ==========================================
# CARGAR DATASET
# ==========================================

df = pd.read_csv("/content/fraudTrain.csv")

# ==========================================
# LIMPIEZA DE DATOS
# ==========================================

# Eliminar columnas innecesarias
drop_cols = [
    'Unnamed: 0',
    'trans_date_trans_time',
    'first',
    'last',
    'street',
    'job',
    'dob'
]

df.drop(columns=drop_cols, inplace=True)

# Eliminar filas con NaN en la columna 'is_fraud'
df.dropna(subset=['is_fraud'], inplace=True)

# ==========================================
# CODIFICAR VARIABLES CATEGÓRICAS
# ==========================================

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# ==========================================
# VARIABLES
# ==========================================

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

# ==========================================
# DIVISIÓN DE DATOS
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# MODELOS BASE
# ==========================================

estimators = [

    ('rf', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )),

    ('xgb', XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))

]

# ==========================================
# META MODELO
# ==========================================

stack_model = StackingClassifier(

    estimators=estimators,

    final_estimator=LogisticRegression(),

    cv=5

)

# ==========================================
# ENTRENAMIENTO
# ==========================================

stack_model.fit(X_train, y_train)

# ==========================================
# PREDICCIONES
# ==========================================

y_pred = stack_model.predict(X_test)

# ==========================================
# EVALUACIÓN
# ==========================================

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9974918165200017

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     70111
         1.0       0.87      0.72      0.79       458

    accuracy                           1.00     70569
   macro avg       0.93      0.86      0.89     70569
weighted avg       1.00      1.00      1.00     70569


Matriz de Confusión:
[[70060    51]
 [  126   332]]


In [4]:
import joblib

# Guardar el modelo entrenado
joblib.dump(stack_model, 'stack_model.joblib')
print("Modelo stack_model guardado exitosamente como stack_model.joblib")

Modelo stack_model guardado exitosamente como stack_model.joblib


## Prueba de Inferencia con el Modelo Exportado

Demostración cómo cargar el modelo guardado y usarlo para hacer predicciones en una nueva transacción. Es CRÍTICO recordar que cualquier nueva transacción que se envíe al modelo para inferencia debe pasar por **exactamente los mismos pasos de preprocesamiento** (eliminación de columnas, codificación de categóricas con los mismos `LabelEncoder`s, etc.) que los datos de entrenamiento.

Para esta demostración, tomaremos una transacción de ejemplo del conjunto `X_test`, ya que ya ha sido preprocesada de la misma manera que los datos de entrenamiento.

In [5]:


import joblib
import pandas as pd
from flask import Flask, request, jsonify

# --- 1. Cargar el Modelo Entrenado ---
# Se carga el modelo una sola vez al iniciar la aplicación para optimizar el rendimiento.
try:
    loaded_stack_model = joblib.load('stack_model.joblib')
    print("Modelo 'stack_model.joblib' cargado exitosamente.")
except FileNotFoundError:
    print("Error: 'stack_model.joblib' no encontrado. Asegúrate de haber exportado el modelo.")
    # En un entorno real,
    loaded_stack_model = None # Asegurarse de que el modelo no sea None si no se carga

#


# --- 2. Inicializar la Aplicación Flask ---
app = Flask(__name__)

# --- 3. Definir el Endpoint de Predicción ---
@app.route('/predict', methods=['POST'])
def predict():
    if loaded_stack_model is None:
        return jsonify({"error": "Modelo no cargado. Por favor, verifica el log de inicio."}), 500

    # Obtener los datos de la solicitud POST
    data = request.get_json(force=True)

    # Convertir el JSON de entrada a un DataFrame de pandas
    # Aseguramos que el DataFrame tenga una sola fila y las columnas correctas.
    try:
        # Convertimos el diccionario a un DataFrame y luego lo transponemos para que sea una fila
        # Si el input ya viene como lista de diccionarios, pd.DataFrame(data) funciona directamente.
        # Para este ejemplo, asumimos un solo diccionario por solicitud.
        input_df = pd.DataFrame([data])

        # --- Preprocesamiento en la API (CRÍTICO) ---
        # Aquí es donde aplicarías las mismas transformaciones que en el entrenamiento.
        # Si recibieras datos RAW, harías cosas como:
        # 1. Eliminar columnas innecesarias.
        # 2. Convertir tipos de datos si es necesario.
        # 3. Aplicar los LabelEncoders guardados a las columnas categóricas:
        #    input_df['gender'] = loaded_le_gender.transform(input_df['gender'])
        #    input_df['category'] = loaded_le_category.transform(input_df['category'])
        #    ...
        #    Por simplicidad en este ejemplo, ASUMIMOS que 'data' ya viene preprocesado
        #    y listo para el modelo (como un fragmento de X_test).

    except Exception as e:
        return jsonify({"error": f"Error en el preprocesamiento de los datos de entrada: {e}"}), 400

    # Realizar la predicción
    prediction = loaded_stack_model.predict(input_df)
    prediction_proba = loaded_stack_model.predict_proba(input_df)

    # Devolver la predicción en formato JSON
    response = {
        'prediction': int(prediction[0]), # Convertir a int para JSON serializable
        'probability_no_fraud': float(prediction_proba[0][0]),
        'probability_fraud': float(prediction_proba[0][1])
    }

    return jsonify(response)

# --- 4. Ejecutar la Aplicación Flask ---
if __name__ == '__main__':

    print("\nPara ejecutar esta API:")
    print("1. Guarda este código en un archivo Python.")
    print("2. Abrir  terminal y navega al directorio donde guardaste 'app.py'.")
    print("3. Ejecuta el comando: 'python app.py'")
    print("La API estará disponible en http://127.0.0.1:5000/predict (para solicitudes POST).")
    # app.run(debug=True, host='0.0.0.0', port=5000) # Descomentar para ejecutar si es posible.

In [6]:
import joblib
import pandas as pd

# Cargar el modelo guardado
loaded_stack_model = joblib.load('stack_model.joblib')
print("Modelo 'stack_model.joblib' cargado exitosamente.")

# Tomar una transacción de ejemplo del conjunto de prueba (X_test ya está preprocesado)
sample_transaction = X_test.iloc[[0]] # Usamos .iloc[[0]] para mantener el DataFrame original con una fila

print(f"\nTransacción de ejemplo (primer elemento de X_test):\n{sample_transaction}")

# Realizar una predicción
prediction = loaded_stack_model.predict(sample_transaction)
prediction_proba = loaded_stack_model.predict_proba(sample_transaction)

print(f"\nPredicción (0: No Fraude, 1: Fraude): {prediction[0]}")
print(f"Probabilidades de predicción (No Fraude, Fraude): {prediction_proba[0]}")

if prediction[0] == 1:
    print("\n=> ¡Alerta de fraude para esta transacción!")
else:
    print("\n=> Transacción legítima detectada.")

### Interpretación Detallada de las Métricas y la Matriz de Confusión

Los resultados de la evaluación nos proporcionan una visión clara del rendimiento de nuestro modelo de Stacking en la detección de fraude. Analicemos cada parte en detalle:

#### 1. Accuracy (Precisión Global)

**`Accuracy: 0.9971145919208574`**

*   **Interpretación:** El accuracy mide la proporción de predicciones correctas sobre el total de predicciones. Un valor de 0.9971 (aproximadamente 99.71%) es extremadamente alto y sugiere que el modelo es, en general, muy bueno clasificando las transacciones. Sin embargo, en problemas de detección de fraude (donde las clases están desbalanceadas, es decir, hay muchas más transacciones no fraudulentas que fraudulentas), el accuracy por sí solo puede ser engañoso. Un modelo que predice "no fraude" todo el tiempo podría tener un accuracy alto si el fraude es muy raro.

#### 2. Classification Report (Informe de Clasificación)

El informe de clasificación desglosa el rendimiento para cada clase (0.0 para no fraude, 1.0 para fraude).

```
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     38501
         1.0       0.88      0.75      0.81       315

    accuracy                           1.00     38816
   macro avg       0.94      0.87      0.90     38816
weighted avg       1.00      1.00      1.00     38816
```

*   **Clase 0.0 (No Fraude):**
    *   **Precision (1.00):** De todas las transacciones que el modelo predijo como NO fraudulentas, el 100% fueron realmente NO fraudulentas. Esto es excelente.
    *   **Recall (1.00):** De todas las transacciones que eran realmente NO fraudulentas, el modelo identificó el 100% correctamente. Esto también es excelente.
    *   **F1-score (1.00):** Es la media armónica de precisión y recall. Un F1-score de 1.00 confirma el rendimiento perfecto para la clase no fraudulenta.
    *   **Support (38501):** Número de instancias reales de la clase no fraudulenta en el conjunto de prueba.

*   **Clase 1.0 (Fraude):**
    *   **Precision (0.88):** De todas las transacciones que el modelo predijo como FRAUDULENTAS, el 88% fueron realmente fraudulentas. Esto significa que un 12% de las veces que el modelo dice que hay fraude, se equivoca (falsos positivos).
    *   **Recall (0.75):** De todas las transacciones que eran realmente FRAUDULENTAS, el modelo identificó el 75% correctamente. Esto significa que el modelo dejó escapar un 25% de los fraudes reales (falsos negativos).
    *   **F1-score (0.81):** Refleja un equilibrio razonable entre precisión y recall para la clase de fraude. Es una buena métrica para clases desbalanceadas.
    *   **Support (315):** Número de instancias reales de la clase fraudulenta en el conjunto de prueba.

*   **Métricas Promedio:**
    *   **macro avg:** Calcula la media no ponderada de las métricas por clase. Da el mismo peso a ambas clases, lo cual es útil en clases desbalanceadas.
    *   **weighted avg:** Calcula la media ponderada por el número de instancias de cada clase. Como hay muchas más instancias de no fraude, el promedio ponderado se acerca mucho a las métricas de la clase no fraudulenta.

#### 3. Matriz de Confusión

```
Matriz de Confusión:
[[38468    33]
 [   79   236]]
```

La matriz de confusión se lee de la siguiente manera:

|                   | Predicho No Fraude | Predicho Fraude |
| :---------------- | :----------------- | :-------------- |
| **Real No Fraude**| 38468 (Verdaderos Negativos) | 33 (Falsos Positivos) |
| **Real Fraude**   | 79 (Falsos Negativos)   | 236 (Verdaderos Positivos) |

*   **Verdaderos Negativos (TN = 38468):** El modelo predijo correctamente 38,468 transacciones como NO fraudulentas cuando realmente no lo eran. Esto es muy bueno.

*   **Falsos Positivos (FP = 33):** El modelo predijo incorrectamente 33 transacciones como FRAUDULENTAS cuando en realidad NO lo eran. Estos son errores de Tipo I. En un contexto de fraude, un falso positivo puede significar que se rechaza una transacción legítima o se genera una alerta innecesaria.

*   **Falsos Negativos (FN = 79):** El modelo predijo incorrectamente 79 transacciones como NO fraudulentas cuando en realidad SÍ lo eran. Estos son errores de Tipo II. En un contexto de fraude, un falso negativo es el error más costoso, ya que significa que el fraude no fue detectado y podría resultar en pérdidas económicas para la entidad afectada.

*   **Verdaderos Positivos (TP = 236):** El modelo predijo correctamente 236 transacciones como FRAUDULENTAS cuando realmente lo eran. Esto indica que el modelo es capaz de identificar una parte significativa de los fraudes.

#### Conclusión de la Interpretación

El modelo es excepcionalmente bueno para identificar transacciones legítimas. Sin embargo, en la detección de fraude, los **falsos negativos (79)** son una preocupación principal. Aunque el modelo detectó 236 casos de fraude, aún dejó pasar 79. Dependiendo del costo asociado a un falso negativo (fraude no detectado) versus el costo de un falso positivo (alerta innecesaria), podríamos necesitar ajustar el umbral de decisión del modelo o explorar modelos que prioricen un recall más alto para la clase de fraude, incluso si eso significa una pequeña disminución en la precisión.

Este notebook implementa un **Stacking Ensemble** para la detección de fraude. El Stacking es una técnica avanzada de ensemble que combina las predicciones de varios modelos base (llamados *estimators*) y utiliza un meta-modelo (o *final_estimator*) para hacer la predicción final. Esto permite aprovechar las fortalezas de diferentes algoritmos para mejorar el rendimiento general del modelo.

### Librerías
Aquí se importan todas las bibliotecas necesarias para la tarea:

*   **`pandas`**: Para la manipulación y análisis de datos, especialmente con DataFrames.
*   **`numpy`**: Para operaciones numéricas, a menudo utilizado en conjunto con pandas.
*   **`sklearn.model_selection.train_test_split`**: Para dividir los datos en conjuntos de entrenamiento y prueba.
*   **`sklearn.preprocessing.LabelEncoder`**: Para codificar variables categóricas a valores numéricos.
*   **`sklearn.ensemble.RandomForestClassifier`**: Un algoritmo de ensemble basado en árboles de decisión, usado como modelo base.
*   **`sklearn.ensemble.StackingClassifier`**: La clase principal para construir el modelo de Stacking.
*   **`sklearn.linear_model.LogisticRegression`**: Un modelo lineal simple, utilizado aquí como meta-modelo.
*   **`xgboost.XGBClassifier`**: Otro potente algoritmo de ensemble basado en gradient boosting, usado como modelo base.
*   **`sklearn.metrics`**: Para evaluar el rendimiento del modelo (`accuracy_score`, `classification_report`, `confusion_matrix`).

### Cargar Dataset
Se carga el conjunto de datos de entrenamiento de fraude desde un archivo CSV llamado `fraudTrain.csv` en un DataFrame de pandas.

### Limpieza de Datos
Se identifican y eliminan columnas que no son relevantes para el modelo o que pueden ser redundantes. En este caso, columnas como `Unnamed: 0` (índice de archivo), `trans_date_trans_time` (ya que se puede extraer información de fecha/hora si fuera necesario), y datos personales como `first`, `last`, `street`, `job`, `dob` se eliminan para simplificar el modelo y evitar posibles sesgos o fugas de información.

### Codificar Variables Categóricas
Se utiliza `LabelEncoder` para transformar las variables categóricas (columnas con tipo de dato 'object') en representaciones numéricas. Esto es necesario porque la mayoría de los algoritmos de machine learning requieren entradas numéricas. Cada valor único en una columna categórica se asigna a un número entero.

### Variables
Se definen las variables predictoras (`X`) y la variable objetivo (`y`).

*   `X`: Contiene todas las columnas del DataFrame excepto la columna 'is_fraud', que es la variable objetivo que queremos predecir.
*   `y`: Contiene únicamente la columna 'is_fraud', que indica si una transacción es fraudulenta (1) o no (0).

### División de Datos
Los datos se dividen en conjuntos de entrenamiento (`X_train`, `y_train`) y prueba (`X_test`, `y_test`).

*   `test_size=0.2`: El 20% de los datos se utilizará para la prueba y el 80% para el entrenamiento.
*   `random_state=42`: Asegura que la división de los datos sea la misma cada vez que se ejecuta el código, lo que permite la reproducibilidad.

### Modelos Base
Se definen los modelos base que formarán parte del ensemble Stacking:

*   **`RandomForestClassifier`**: Un bosque aleatorio con 100 árboles (`n_estimators=100`). Es robusto y bueno para manejar datos complejos.
*   **`XGBClassifier`**: Un clasificador de *Gradient Boosting* que es muy eficiente y de alto rendimiento. Se configura con 100 estimadores (`n_estimators=100`), una tasa de aprendizaje de 0.1 (`learning_rate=0.1`) y una profundidad máxima de árbol de 5 (`max_depth=5`).

Ambos modelos usan `random_state=42` para garantizar la reproducibilidad.

### Meta Modelo
Se inicializa el `StackingClassifier`.

*   **`estimators=estimators`**: Se le pasan los modelos base definidos anteriormente.
*   **`final_estimator=LogisticRegression()`**: Se define el meta-modelo, que aprenderá a combinar las predicciones de los modelos base. En este caso, se usa una Regresión Logística.
*   **`cv=5`**: Se usa validación cruzada con 5 pliegues durante el entrenamiento del Stacking, lo que ayuda a evitar el sobreajuste del meta-modelo a las predicciones de los modelos base.

# DESARROLLO MEJORAMIENTO DEL MODELO


In [7]:
# =====================================================
# LIBRERÍAS PRINCIPALES
# =====================================================

# Manipulación de datos
import pandas as pd
import numpy as np

# Dividir datos
from sklearn.model_selection import train_test_split

# Codificar variables categóricas
from sklearn.preprocessing import LabelEncoder

# Modelos base
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Modelo stacking
from sklearn.ensemble import StackingClassifier

# Meta-modelo
from sklearn.linear_model import LogisticRegression

# Métricas
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# =====================================================
# CARGAR DATASET
# =====================================================

# Leer archivo CSV
df = pd.read_csv("/content/fraudTrain.csv")

# Mostrar primeras filas
print(df.head())

 **En esta parte reducimos el data set a un 40% ya que el original contiene
demaciada información asi concluimos un acertamiento mas preciso de fraude.**

In [ ]:
# =====================================================
# REDUCIR DATASET
# =====================================================

df_small, _ = train_test_split(

    df,

    train_size=0.4,

    stratify=df['is_fraud'],

    random_state=42
)

# Reemplazar dataset
df = df_small

# Mostrar tamaño nuevo
print("Nuevo tamaño:")
print(df.shape)

In [ ]:
# ======================================================
# INFORMACIÓN GENERAL
# ======================================================

# Información del dataset
print(df.info())

# Valores nulos
print(df.isnull().sum())

In [ ]:
# =====================================================
# LIMPIEZA DE DATOS
# =====================================================

columnas_eliminar = [

    'Unnamed: 0',

    'trans_date_trans_time',

    'first',

    'last',

    'street',

    'job',

    'dob'
]

# Eliminar columnas
df.drop(columns=columnas_eliminar, inplace=True)

# Mostrar columnas restantes
print(df.columns)

In [ ]:
# =====================================================
# CODIFICAR VARIABLES
# =====================================================

# Crear LabelEncoder
label_encoder = LabelEncoder()

# Recorrer columnas tipo texto
for columna in df.select_dtypes(include='object').columns:

    # Convertir texto a números
    df[columna] = label_encoder.fit_transform(df[columna])

# Mostrar primeras filas
print(df.head())

In [ ]:
# =====================================================
# VARIABLES PREDICTORAS Y OBJETIVO
# =====================================================

# Variables predictoras
X = df.drop('is_fraud', axis=1)

# Variable objetivo
y = df['is_fraud']

In [ ]:
# =====================================================
# DIVISIÓN DE DATOS
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    stratify=y,

    random_state=42
)

In [ ]:
# =====================================================
# BALANCEO DE CLASES
# =====================================================

# Cantidad de fraudes
fraud = y_train.value_counts()[1]

# Cantidad de no fraude
normal = y_train.value_counts()[0]

# Mostrar cantidades
print("No fraude:", normal)
print("Fraude:", fraud)

# Peso moderado
peso = 10

print("Peso usado:", peso)

Tambien utilizamos XGBOOSTpara poder clasificar nuestro modelo.

In [ ]:
# =====================================================
# MODELO XGBOOST
# =====================================================

modelo = XGBClassifier(

    # Cantidad árboles
    n_estimators=120,

    # Profundidad árboles
    max_depth=5,

    # Velocidad aprendizaje
    learning_rate=0.05,

    # Usa parte filas
    subsample=0.8,

    # Usa parte columnas
    colsample_bytree=0.8,

    # Evita overfitting
    min_child_weight=3,

    # Árboles más conservadores
    gamma=1,

    # Balancear fraude
    scale_pos_weight=peso,

    # Reproducibilidad
    random_state=42,

    # Métrica interna
    eval_metric='logloss',

    # Usar todos los núcleos
    n_jobs=-1
)

In [ ]:
# =====================================================
# ENTRENAMIENTO
# =====================================================

modelo.fit(

    X_train,

    y_train,

    eval_set=[(X_test, y_test)],

    verbose=True
)

Esta imagen nos muestra como se entreno el modelo XGBoost con 120 árboles, El modelo XGBoost se estuvo entrenando y en cada ronda cometía menos errores, terminando con un error muy bajo de 0.02123, lo que significa que el modelo aprendió muy bien a detectar fraudes.

In [ ]:
# =====================================================
# PROBABILIDADES
# =====================================================

# Obtener probabilidades
y_prob = modelo.predict_proba(X_test)[:,1]

print(y_prob[:10])

In [ ]:
# =====================================================
# THRESHOLD PERSONALIZADO
# =====================================================

# Threshold
threshold = 0.80

# Convertir probabilidades
y_pred = (y_prob >= threshold).astype(int)

In [ ]:
# =====================================================
# EVALUACIÓN
# =====================================================

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:")

print(accuracy)

# Reporte clasificación
print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

# Matriz confusión
print("\nMatriz de Confusión:\n")

matriz = confusion_matrix(y_test, y_pred)

print(matriz)

**Mi modelo clasifico 99.63% de todas las transacciones.**



# Clase 0 (No Fraude):

**Precision (1.00):** De todas las transacciones que el modelo predijo como NO fraudulentas, el 100% fueron realmente NO fraudulentas.

**Recall (1.00):**  De todas las transacciones que eran realmente NO fraudulentas, el modelo identificó el 100% correctamente.

**F1-score (1.00):** Es la media armónica de precisión y recall. Un F1-score de 1.00 confirma el rendimiento perfecto para la clase no fraudulenta.

**Support (103134):** Número de instancias reales de la clase no fraudulenta en el conjunto de prueba.

# Clase 1 (Fraude):

**Precision (0.68):** De todas las transacciones que el modelo predijo como FRAUDULENTAS, el 68% fueron realmente fraudulentas. Esto significa que un 32% de las veces que el modelo dice que hay fraude, se equivoca (falsos positivos).

**Recall ( 0.67):** De todas las transacciones que eran realmente FRAUDULENTAS, el modelo identificó el 67% correctamente. Esto significa que el modelo dejó escapar un 33% de los fraudes reales (falsos negativos).

**F1-score (0.67):** Indica un equilibrio adecuado entre precisión y recall en la detección de fraude, siendo una métrica importante para evaluar modelos en datasets desbalanceados.

**Support (600):** Número de instancias reales de la clase fraudulenta en el conjunto de prueba.

# MATRIZ DE CONFUSIÓN

**VERDADEROS NEGATIVOS:** 102,942 transacciones normales, como no fraudulentas.

**FALSOS POSITIVOS:** 192 transacciones normales, fraude

**FALSOS NEGATIVOS:** 197 fraudes reales, transacciones normales

**VERDADEROS POSITIVOS**: 403 fraudes reales

# CONCLUSIÓN

La matriz de confusión evidencia que el modelo Boosting presenta un desempeño sólido y equilibrado en la detección de fraude financiero.

El sistema logró clasificar correctamente la mayoría de las transacciones legítimas y detectar una cantidad considerable de fraudes reales, manteniendo un número relativamente bajo de falsas alarmas.

Aunque aún existen fraudes no detectados (falsos negativos), el modelo representa una solución robusta y eficiente para escenarios reales de detección de fraude financiero.

In [ ]:
# =====================================================
# MATRIZ VISUAL
# =====================================================

plt.figure(figsize=(7,5))

sns.heatmap(

    matriz,

    annot=True,

    fmt='d',

    cmap='Blues'
)

plt.title("Matriz de Confusión")

plt.xlabel("Predicción")

plt.ylabel("Valor Real")

plt.show()

 Este modelo podemos identificar correctamente 102,942 transacciones normales y 403 fraudes reales, pero se equivocó en 192 transacciones normales clasificándolas como fraude y en 197 fraudes reales que no detectó.


In [ ]:
# =====================================================
# IMPORTANCIA VARIABLES
# =====================================================

from xgboost import plot_importance

plt.figure(figsize=(10,8))

plot_importance(modelo)

plt.show()

El gráfico muestra la importancia de cada variable en un modelo de detección de fraude. amt (monto) y category (categoría de transacción) son las variables más importantes con puntajes de 879 y 863, siendo los principales predictores de fraude. El resto de variables como tiempo, población de la ciudad y número de tarjeta tienen importancia significativamente menor.